# Q1

## Q1.png

Refer to q1.png

## Q1.sql

Refer to q1.sql 

# Q2

## Q2 Code

In [1]:
import feedparser
import psycopg2
from bs4 import BeautifulSoup

url = "https://news.google.com/rss/search?q=technology&hl=en-US&gl=US&ceid=US:en"

feed = feedparser.parse(url)
lastBuildDate = feed.feed.get("updated", None)

conn = psycopg2.connect(
    host="localhost",
    database="postgres",   
    user="postgres",       
    password="12345"       
)
cur = conn.cursor()

cur.execute("DELETE FROM news.google_news;")

for entry in feed.entries:
    title = entry.get("title")
    link = entry.get("link")
    pubDate = entry.get("published")
    raw_desc = entry.get("description", "")
    description = BeautifulSoup(raw_desc, "html.parser").get_text()
    source = entry.source.title if "source" in entry and hasattr(entry.source, "title") else None
    cur.execute("""INSERT INTO news.google_news (lastBuildDate, title, link, pubDate, description, source)VALUES (%s, %s, %s, %s, %s, %s)""", (lastBuildDate, title, link, pubDate, description, source))

conn.commit()
cur.close()
conn.close()

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("hmk2_q2").getOrCreate()


In [3]:
df = spark.read.format("jdbc").options(
    url="jdbc:postgresql://localhost:5432/postgres",
    driver="org.postgresql.Driver",
    dbtable="news.google_news",
    user="postgres",
    password="12345"
).load()

df.createOrReplaceTempView("google_news")

result = spark.sql("SELECT * FROM google_news ORDER BY random() LIMIT 5")
result.show()


+----+-------------------+--------------------+--------------------+-------------------+--------------------+------------------+--------+
|  id|      lastbuilddate|               title|                link|            pubdate|         description|            source|category|
+----+-------------------+--------------------+--------------------+-------------------+--------------------+------------------+--------+
|1588|2025-09-19 00:57:54|Should You Think ...|https://news.goog...|2025-09-18 22:10:53|Should You Think ...|     simplywall.st|    NULL|
|1522|2025-09-19 00:57:54|Policy Scholars p...|https://news.goog...|2025-09-18 17:30:25|Policy Scholars p...|Virginia Tech News|    NULL|
|1526|2025-09-19 00:57:54|Wells Technology ...|https://news.goog...|2025-09-18 17:49:53|Wells Technology ...|   Bemidji Pioneer|    NULL|
|1563|2025-09-19 00:57:54|How technology is...|https://news.goog...|2025-09-18 22:21:00|How technology is...|          CBS News|    NULL|
|1572|2025-09-19 00:57:54|Tech Pio

At this stage, the table had no category column before. But i restart the Kernal and run all the cells, so it turns out to have a category column now.

## Q2.png

Refer to q2.png

# Q3

## Q3 code

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("hmk2_q3").getOrCreate()

In [5]:
df = spark.read.format("jdbc").options(
    url="jdbc:postgresql://localhost:5432/postgres",
    driver="org.postgresql.Driver",
    dbtable="news.google_news",
    user="postgres",
    password="12345"
).load()

df.createOrReplaceTempView("google_news")

In [6]:
def last24_news():
    query = """
        SELECT id, title, source, pubDate
        FROM google_news
        WHERE pubDate >= current_timestamp() - interval 24 hours
        ORDER BY pubDate DESC
    """
    return spark.sql(query)

## Q3-output

In [7]:
news_last24 = last24_news()
news_last24.show()

+----+--------------------+--------------------+-------------------+
|  id|               title|              source|            pubDate|
+----+--------------------+--------------------+-------------------+
|1518|GoPro Wins Emmy® ...|         PR Newswire|2025-09-18 23:51:00|
|1572|Tech Pioneer GoPr...|         Stock Titan|2025-09-18 23:51:00|
|1566|Men's Golf Take 4...|Stevens Institute...|2025-09-18 23:29:20|
|1555|How Intel's Nvidi...|             Reuters|2025-09-18 22:58:47|
|1509|Nvidia spent over...|             Reuters|2025-09-18 22:58:07|
|1530|Ochsner Health Of...|            Newswise|2025-09-18 22:45:00|
|1548|Arlington's Retro...|City of Arlington...|2025-09-18 22:41:17|
|1508|Nvidia spent over...|       Yahoo Finance|2025-09-18 22:35:52|
|1507|Nvidia just spent...|                CNBC|2025-09-18 22:23:04|
|1563|How technology is...|            CBS News|2025-09-18 22:21:00|
|1588|Should You Think ...|       simplywall.st|2025-09-18 22:10:53|
|1605|Sapient Insights ...|       

Refer to q3.png

# Q4

## Q4.sql

Refer to the q4.sql

## Q4 code

In [8]:
import feedparser
import psycopg2
from bs4 import BeautifulSoup

conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="12345"
)
cur = conn.cursor()

def insert_feed(url, category):
    feed = feedparser.parse(url)
    for entry in feed.entries:
        title = entry.get("title")
        link = entry.get("link")
        pubDate = entry.get("published")
        raw_desc = entry.get("description", "")
        description = BeautifulSoup(raw_desc, "html.parser").get_text()
        if "source" in entry and hasattr(entry.source, "title"):
            source = entry.source.title
        else:
            source = None

        cur.execute(
            "INSERT INTO news.google_news (title, link, pubDate, description, source, category) "
            "VALUES (%s, %s, %s, %s, %s, %s)",
            (title, link, pubDate, description, source, category)
        )

    conn.commit()

insert_feed("https://news.google.com/rss/search?q=technology&hl=en-US&gl=US&ceid=US:en", "technology")
insert_feed("https://news.google.com/rss/search?q=business&hl=en-US&gl=US&ceid=US:en", "business")
insert_feed("https://news.google.com/rss/search?q=sports&hl=en-US&gl=US&ceid=US:en", "sports")

cur.close()
conn.close()

In [9]:
df = spark.read.format("jdbc").options(
    url="jdbc:postgresql://localhost:5432/postgres",
    driver="org.postgresql.Driver",
    dbtable="news.google_news",
    user="postgres",
    password="12345"
).load()

df.createOrReplaceTempView("google_news")


## Q4-output

In [10]:
result = spark.sql("SELECT DISTINCT category FROM google_news")
result.show()

+----------+
|  category|
+----------+
|technology|
|    sports|
|  business|
|      NULL|
+----------+



# Q5

## Q5 code

In [16]:
from pyspark.sql.functions import col, lower

df = spark.read.format("jdbc").options(
    url="jdbc:postgresql://localhost:5432/postgres",
    driver="org.postgresql.Driver",
    dbtable="news.google_news",
    user="postgres",
    password="12345"
).load()

In [17]:
df_clean = df.filter(~lower(col("title")).like("%nfl%"))

In [18]:
df_clean.write.format("jdbc").options(
    url="jdbc:postgresql://localhost:5432/postgres",
    driver="org.postgresql.Driver",
    dbtable="news.google_news",
    user="postgres",
    password="12345"
).mode("overwrite").save()

## Q5-output

In [19]:
result = spark.sql("""
    SELECT * FROM google_news
    WHERE LOWER(title) LIKE '%nfl%'
""")
result.show(truncate=False)

+---+-------------+-----+----+-------+-----------+------+--------+
|id |lastbuilddate|title|link|pubdate|description|source|category|
+---+-------------+-----+----+-------+-----------+------+--------+
+---+-------------+-----+----+-------+-----------+------+--------+



Refer to q5.png

# Q6

Refer to q6.png